In [1]:
import pandas as pd
import datetime as dt
from sklearn.preprocessing import StandardScaler

In [2]:
pd.set_option("display.max_columns",None)
pd.set_option("display.width",600)
pd.set_option("display.max_rows",600)
pd.set_option("display.float_format",lambda x:"%.2f" %x)

In [3]:
df = pd.read_csv("C:\\Users\\Ali Can\\Desktop\\Proje\\DinamikProje\\data\\interim\\intermadiate_data_after_data_analysis.csv")
df.head(10)

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Target,Failure Type,Date
0,M,298.10,308.60,1551,42.80,0,0,No Failure,2024-01-01 00:00:00
1,L,298.20,308.70,1408,46.30,3,0,No Failure,2024-01-01 00:15:00
2,L,298.10,308.50,1498,49.40,5,0,No Failure,2024-01-01 00:30:00
3,L,298.20,308.60,1433,39.50,7,0,No Failure,2024-01-01 00:45:00
4,L,298.20,308.70,1408,40.00,9,0,No Failure,2024-01-01 01:00:00
5,M,298.10,308.60,1425,41.90,11,0,No Failure,2024-01-01 01:15:00
6,L,298.10,308.60,1558,42.40,14,0,No Failure,2024-01-01 01:30:00
7,L,298.10,308.60,1527,40.20,16,0,No Failure,2024-01-01 01:45:00
8,M,298.30,308.70,1667,28.60,18,0,No Failure,2024-01-01 02:00:00
9,M,298.50,309.00,1741,28.00,21,0,No Failure,2024-01-01 02:15:00


In [4]:
df.dtypes

Type                        object
Air temperature [K]        float64
Process temperature [K]    float64
Rotational speed [rpm]       int64
Torque [Nm]                float64
Tool wear [min]              int64
Target                       int64
Failure Type                object
Date                        object
dtype: object

In [5]:
df["Date"].dtype

dtype('O')

In [6]:
df["Date"] = pd.to_datetime(df["Date"],format="%Y-%m-%d")

In [7]:
df["Date"].dtype

dtype('<M8[ns]')

In [8]:
df["Year"] = df["Date"].dt.year

In [9]:
df["Month"] = df["Date"].dt.month

In [10]:
df["Day"] = df["Date"].dt.day

In [11]:
df["Weekday"] = df["Date"].dt.weekday

In [12]:
df.loc[df["Weekday"] == 0,"Weekday"] = "Monday"
df.loc[df["Weekday"] == 1,"Weekday"] = "Tuesday"
df.loc[df["Weekday"] == 2,"Weekday"] = "Wednesday"
df.loc[df["Weekday"] == 3,"Weekday"] = "Thursday"
df.loc[df["Weekday"] == 4,"Weekday"] = "Friday"
df.loc[df["Weekday"] == 5,"Weekday"] = "Saturday"
df.loc[df["Weekday"] == 6,"Weekday"] = "Sunday"

In [13]:
df['Temperature Difference'] = df['Process temperature [K]'] - df['Air temperature [K]']

In [14]:
df['Power [W]'] = (df['Torque [Nm]'] * df['Rotational speed [rpm]']) / 9.5488

In [15]:
bins = [0, 50, 100, 150, 200, 244]
labels = ['Very Low', 'Low', 'Medium', 'High', 'Very High']
df['Wear Category'] = pd.cut(df['Tool wear [min]'], bins=bins, labels=labels, include_lowest=True)

In [16]:
df["Wear Category"].value_counts()

Very Low     2351
Medium       2257
High         2248
Low          2241
Very High     732
Name: Wear Category, dtype: int64

In [17]:
df['RPM_Torque_Ratio'] = df['Rotational speed [rpm]'] / df['Torque [Nm]'] 

In [18]:
df["Air_Process_Interaction"] = df["Air temperature [K]"] * df["Process temperature [K]"]

In [19]:
df['Temperature_Wear_Interaction'] = df['Process temperature [K]'] * df['Tool wear [min]']

In [20]:
df["ToolWear_roll_mean_1_hours"] = df["Tool wear [min]"].shift(1).rolling(window=4).mean()

In [21]:
df["ToolWear_roll_mean_6_hours"] = df["Tool wear [min]"].shift(1).rolling(window=24).mean()

In [22]:
df["ToolWear_roll_mean_12_hours"] = df["Tool wear [min]"].shift(1).rolling(window=48).mean()

In [23]:
df["ToolWear_roll_mean_24_hours"] = df["Tool wear [min]"].shift(1).rolling(window=96).mean()

In [24]:
df.drop(index=df.iloc[0:96].index,axis=0,inplace=True)

In [25]:
df.to_csv("C:\\Users\\Ali Can\\Desktop\\Proje\\DinamikProje\\data\\interim\\intermediate_data_after_feature_engineering.csv",index=False)

In [26]:
df.drop(["Date"],axis=1,inplace=True)

In [27]:
df.drop(["Target"],axis=1,inplace=True)

In [28]:
def degisken_turlerini_ayır(dataframe,cat_th=10,car_th=20): #This function separates the columns of the dataframe into their types
    
    kategorik_degiskenler = [col for col in dataframe.columns if dataframe[col].dtype == 'O']
    sayisal_degiskenler = [col for col in dataframe.columns if dataframe[col].dtype != 'O']
    
    kat_ama_car = [col for col in dataframe.columns if dataframe[col].dtype == 'O' and dataframe[col].nunique() > car_th]
    say_ama_cat = [col for col in dataframe.columns if dataframe[col].dtype != 'O' and dataframe[col].nunique() < cat_th]
    
    kategorik_degiskenler = kategorik_degiskenler + say_ama_cat
    
    kategorik_degiskenler = [col for col in kategorik_degiskenler if col not in kat_ama_car]
    
    sayisal_degiskenler = [col for col in sayisal_degiskenler if col not in say_ama_cat]
    
    return kategorik_degiskenler,sayisal_degiskenler,kat_ama_car

In [29]:
kategorik_degiskenler,sayisal_degiskenler,kat_ama_car = degisken_turlerini_ayır(df)

In [30]:
kategorik_degiskenler

['Type', 'Failure Type', 'Weekday', 'Year', 'Month', 'Wear Category']

In [31]:
sayisal_degiskenler

['Air temperature [K]',
 'Process temperature [K]',
 'Rotational speed [rpm]',
 'Torque [Nm]',
 'Tool wear [min]',
 'Day',
 'Temperature Difference',
 'Power [W]',
 'RPM_Torque_Ratio',
 'Air_Process_Interaction',
 'Temperature_Wear_Interaction',
 'ToolWear_roll_mean_1_hours',
 'ToolWear_roll_mean_6_hours',
 'ToolWear_roll_mean_12_hours',
 'ToolWear_roll_mean_24_hours']

In [32]:
kat_ama_car

[]

In [33]:
kategorik_degiskenler = [col for col in kategorik_degiskenler if col not in ["Failure Type"]]
kategorik_degiskenler

['Type', 'Weekday', 'Year', 'Month', 'Wear Category']

In [34]:
ss = StandardScaler()
df[sayisal_degiskenler] = ss.fit_transform(df[sayisal_degiskenler])

In [35]:
df = pd.get_dummies(data=df,columns=kategorik_degiskenler)

In [36]:
df

,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Failure Type,Day,Temperature Difference,Power [W],RPM_Torque_Ratio,Air_Process_Interaction,Temperature_Wear_Interaction,ToolWear_roll_mean_1_hours,ToolWear_roll_mean_6_hours,ToolWear_roll_mean_12_hours,ToolWear_roll_mean_24_hours,Type_H,Type_L,Type_M,Weekday_Friday,Weekday_Monday,Weekday_Saturday,Weekday_Sunday,Weekday_Thursday,Weekday_Tuesday,Weekday_Wednesday,Year_2024,Month_1,Month_2,Month_3,Month_4,Wear Category_Very Low,Wear Category_Low,Wear Category_Medium,Wear Category_High,Wear Category_Very High
96,-0.56,-0.75,1.49,-1.09,-0.91,No Failure,-1.46,0.00,-0.82,1.09,-0.66,-0.92,-1.07,-1.03,0.26,-1.91,0,0,1,0,0,0,0,0,1,0,1,1,0,0,0,1,0,0,0,0
97,-0.56,-0.82,0.01,-0.80,-0.87,No Failure,-1.46,-0.10,-1.06,0.35,-0.69,-0.87,-1.03,-1.17,0.19,-1.85,0,1,0,0,0,0,0,0,1,0,1,1,0,0,0,0,1,0,0,0
98,-0.61,-0.82,-0.03,0.43,-0.83,No Failure,-1.46,0.00,0.75,-0.44,-0.72,-0.84,-0.98,-1.30,0.13,-1.80,0,1,0,0,0,0,0,0,1,0,1,1,0,0,0,0,1,0,0,0
99,-0.61,-0.75,2.17,-1.61,-0.80,No Failure,-1.46,0.10,-1.44,2.08,-0.69,-0.81,-0.94,-1.44,0.06,-1.74,0,1,0,0,0,0,0,0,1,0,1,1,0,0,0,0,1,0,0,0
100,-0.61,-0.82,3.10,-2.06,-0.77,No Failure,-1.46,0.00,-1.96,3.48,-0.72,-0.78,-0.89,-1.58,-0.01,-1.68,0,1,0,0,0,0,0,0,1,0,1,1,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9824,-0.61,-1.08,0.51,-1.14,-1.48,No Failure,-0.08,-0.40,-1.33,0.83,-0.83,-1.48,-1.64,0.75,1.07,0.06,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,1,0,0,0,0
9825,-0.56,-1.08,0.70,-0.89,-1.43,No Failure,-0.08,-0.50,-0.86,0.64,-0.80,-1.43,-1.60,0.61,1.00,-0.13,1,0,0,0,0,0,1,0,0,0,1,0,0,0,1,1,0,0,0,0
9826,-0.51,-0.95,0.79,-0.73,-1.35,No Failure,-0.08,-0.40,-0.55,0.51,-0.71,-1.36,-1.56,0.47,0.93,-0.32,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,1,0,0,0,0
9827,-0.51,-0.88,-0.80,0.86,-1.31,No Failure,-0.08,-0.30,0.82,-0.78,-0.69,-1.31,-1.51,0.33,0.86,-0.50,1,0,0,0,0,0,1,0,0,0,1,0,0,0,1,1,0,0,0,0


In [37]:
df.drop(["Year_2024"],axis=1,inplace=True)

In [39]:
df

,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Failure Type,Day,Temperature Difference,Power [W],RPM_Torque_Ratio,Air_Process_Interaction,Temperature_Wear_Interaction,ToolWear_roll_mean_1_hours,ToolWear_roll_mean_6_hours,ToolWear_roll_mean_12_hours,ToolWear_roll_mean_24_hours,Type_H,Type_L,Type_M,Weekday_Friday,Weekday_Monday,Weekday_Saturday,Weekday_Sunday,Weekday_Thursday,Weekday_Tuesday,Weekday_Wednesday,Month_1,Month_2,Month_3,Month_4,Wear Category_Very Low,Wear Category_Low,Wear Category_Medium,Wear Category_High,Wear Category_Very High
96,-0.56,-0.75,1.49,-1.09,-0.91,No Failure,-1.46,0.00,-0.82,1.09,-0.66,-0.92,-1.07,-1.03,0.26,-1.91,0,0,1,0,0,0,0,0,1,0,1,0,0,0,1,0,0,0,0
97,-0.56,-0.82,0.01,-0.80,-0.87,No Failure,-1.46,-0.10,-1.06,0.35,-0.69,-0.87,-1.03,-1.17,0.19,-1.85,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0
98,-0.61,-0.82,-0.03,0.43,-0.83,No Failure,-1.46,0.00,0.75,-0.44,-0.72,-0.84,-0.98,-1.30,0.13,-1.80,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0
99,-0.61,-0.75,2.17,-1.61,-0.80,No Failure,-1.46,0.10,-1.44,2.08,-0.69,-0.81,-0.94,-1.44,0.06,-1.74,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0
100,-0.61,-0.82,3.10,-2.06,-0.77,No Failure,-1.46,0.00,-1.96,3.48,-0.72,-0.78,-0.89,-1.58,-0.01,-1.68,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9824,-0.61,-1.08,0.51,-1.14,-1.48,No Failure,-0.08,-0.40,-1.33,0.83,-0.83,-1.48,-1.64,0.75,1.07,0.06,0,0,1,0,0,0,1,0,0,0,0,0,0,1,1,0,0,0,0
9825,-0.56,-1.08,0.70,-0.89,-1.43,No Failure,-0.08,-0.50,-0.86,0.64,-0.80,-1.43,-1.60,0.61,1.00,-0.13,1,0,0,0,0,0,1,0,0,0,0,0,0,1,1,0,0,0,0
9826,-0.51,-0.95,0.79,-0.73,-1.35,No Failure,-0.08,-0.40,-0.55,0.51,-0.71,-1.36,-1.56,0.47,0.93,-0.32,0,0,1,0,0,0,1,0,0,0,0,0,0,1,1,0,0,0,0
9827,-0.51,-0.88,-0.80,0.86,-1.31,No Failure,-0.08,-0.30,0.82,-0.78,-0.69,-1.31,-1.51,0.33,0.86,-0.50,1,0,0,0,0,0,1,0,0,0,0,0,0,1,1,0,0,0,0


In [38]:
df.to_csv("C:\\Users\\Ali Can\\Desktop\\Proje\\DinamikProje\\data\\processed\\processed_data.csv",index=False)